In [1]:

# YR NOVATECH - Task 2
# Data Collection and Preprocessing

!pip -q install ucimlrepo

import pandas as pd
import numpy as np
from ucimlrepo import fetch_ucirepo
import os
import zipfile

# 1. Create project folders
os.makedirs("dataset", exist_ok=True)
os.makedirs("reports", exist_ok=True)

# 2. Collect real-world dataset from UCI
adult = fetch_ucirepo(id=2)

X = adult.data.features
y = adult.data.targets

df = pd.concat([X, y], axis=1)

print("Dataset collected successfully!")
print("Original shape:", df.shape)

# 3. Save original dataset
df.to_csv("dataset/raw_dataset.csv", index=False)

# 4. Clean column names and text values
df.columns = (
    df.columns.str.strip()
    .str.lower()
    .str.replace("-", "_")
)

for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].str.strip()

# 5. Handle missing values
df.replace("?", np.nan, inplace=True)

missing_before = df.isnull().sum().sum()
print("Missing values before cleaning:", missing_before)

# 6. Remove duplicate records
duplicates = df.duplicated().sum()
df.drop_duplicates(inplace=True)

print("Duplicates removed:", duplicates)

# 7. Handle missing numerical and categorical values
numeric_cols = df.select_dtypes(include=np.number).columns
categorical_cols = df.select_dtypes(include="object").columns

for col in numeric_cols:
    df[col] = df[col].fillna(df[col].median())

for col in categorical_cols:
    if df[col].isnull().any():
        df[col] = df[col].fillna(df[col].mode()[0])

# 8. Handle outliers using IQR capping
for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    df[col] = df[col].clip(lower, upper)

# 9. Feature engineering
if "age" in df.columns:
    df["age_group"] = pd.cut(
        df["age"],
        bins=[0, 25, 45, 65, 100],
        labels=["Young", "Adult", "Middle_Aged", "Senior"]
    )

# 10. Save cleaned dataset
df.to_csv("dataset/cleaned_dataset.csv", index=False)

# 11. Generate preprocessing report
report = f"""
YR NOVATECH - TASK 2 REPORT

Project: Data Collection and Preprocessing

Original records: {len(X)}
Original columns: {df.shape[1] - (1 if "age_group" in df.columns else 0)}

Missing values before cleaning: {missing_before}
Duplicate records removed: {duplicates}
Remaining records: {len(df)}
Remaining missing values: {df.isnull().sum().sum()}

Preprocessing completed:
1. Dataset collection
2. Missing value handling
3. Duplicate removal
4. Data type inspection
5. Outlier handling using IQR
6. Feature engineering
7. Cleaned dataset export

Final dataset shape: {df.shape}
"""

with open("reports/preprocessing_report.txt", "w") as f:
    f.write(report)

print(report)

# 12. Create downloadable project ZIP
with zipfile.ZipFile("YR_NOVATECH_Task2.zip", "w") as z:
    z.write("dataset/raw_dataset.csv")
    z.write("dataset/cleaned_dataset.csv")
    z.write("reports/preprocessing_report.txt")

print("PROJECT COMPLETED SUCCESSFULLY!")

from google.colab import files
files.download("YR_NOVATECH_Task2.zip")


Dataset collected successfully!
Original shape: (48842, 15)
Missing values before cleaning: 6465
Duplicates removed: 29

YR NOVATECH - TASK 2 REPORT

Project: Data Collection and Preprocessing

Original records: 48842
Original columns: 15

Missing values before cleaning: 6465
Duplicate records removed: 29
Remaining records: 48813
Remaining missing values: 0

Preprocessing completed:
1. Dataset collection
2. Missing value handling
3. Duplicate removal
4. Data type inspection
5. Outlier handling using IQR
6. Feature engineering
7. Cleaned dataset export

Final dataset shape: (48813, 16)

PROJECT COMPLETED SUCCESSFULLY!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>